In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from codllm.settings.schema import Config
from codllm.data.handler import DataHandler

In [2]:
project_root = Path.cwd().parents[1]
raw = project_root / "data" / "raw" 
processed = project_root / "data" / "processed"

In [3]:
cfg = Config(dataset_size=1.0, data_raw_dir=raw, data_processed_dir=processed)
for source in cfg.data_sources:
    if source.source_id == "historic_strings_en_2024":
        source.enabled = False

cfg.data_sources

[DataSourceConfig(source_id='belgium_1920_1930', path='SOSA_EXTR_1920-1930 (belgium).xlsx', mapping_id='belgium', enabled=True, file_type=None, sep=',', encoding='utf-8', header=0, sheet_name=0, skip_rows=[]),
 DataSourceConfig(source_id='amsterdam_1854_1926', path='AMC_1854_1926_LM.csv', mapping_id='amsterdam', enabled=True, file_type=None, sep=';', encoding='utf-8', header=0, sheet_name=0, skip_rows=[]),
 DataSourceConfig(source_id='copenhagen_may2025', path='Copenhagen_burials_all_May2025.csv', mapping_id='copenhagen', enabled=True, file_type=None, sep=',', encoding='utf-8', header=0, sheet_name=0, skip_rows=[]),
 DataSourceConfig(source_id='ipswich_1871_1911', path='Ipswich_deaths_codllm.txt', mapping_id='ipswich', enabled=True, file_type='csv', sep='|', encoding='latin-1', header=0, sheet_name=0, skip_rows=[]),
 DataSourceConfig(source_id='madrid_1905_1927', path='Madrid 1905_1927.csv', mapping_id='madrid', enabled=True, file_type=None, sep=',', encoding='utf-8', header=0, sheet_n

In [4]:
handler = DataHandler(cfg)
df = handler.ensure_processed(force_reprocess=True)

In [5]:
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

df.head()

Total rows: 1379516
Columns: ['source_id', 'record_id', 'source_path', 'text', 'y_codes', 'label']


,source_id,record_id,source_path,text,y_codes,label
0,belgium_1920_1930,920A3653,/Users/lucasmortensen/Repos/codLLM/data/raw/SO...,cod: Choléra infantile | age: 0.24 | sex: female,[A00.900],A00.900
1,belgium_1920_1930,920A2451,/Users/lucasmortensen/Repos/codLLM/data/raw/SO...,cod: Cholérine | age: 1.21 | sex: male,[A00.900],A00.900
2,belgium_1920_1930,924A2390,/Users/lucasmortensen/Repos/codLLM/data/raw/SO...,cod: Cholérine | age: 62.86 | sex: male,[A00.900],A00.900
3,belgium_1920_1930,930A2361,/Users/lucasmortensen/Repos/codLLM/data/raw/SO...,cod: Cholérine | age: 64.69 | sex: male,[A00.900],A00.900
4,belgium_1920_1930,923A2689,/Users/lucasmortensen/Repos/codLLM/data/raw/SO...,cod: Fièvre muqueuse | age: 21.7 | sex: female,[A01.000],A01.000


In [6]:
splits = handler.split_dataframe(df)
text_col = cfg.dataset_text_column

train_strings = set(splits.train[text_col].fillna("").astype(str).str.strip())
test_strings = set(splits.test[text_col].fillna("").astype(str).str.strip())
train_strings.discard("")
test_strings.discard("")

print(f"Unique train strings: {len(train_strings):,}")
print(f"Unique test strings: {len(test_strings):,}")

Unique train strings: 291,291
Unique test strings: 34,236


In [7]:
shared_strings = train_strings.intersection(test_strings)

print(f"Shared train/test strings: {len(shared_strings):,}")
if test_strings:
    leakage_rate = len(shared_strings) / len(test_strings)
    print(f"Leakage rate in test (unique-string basis): {leakage_rate:.2%}")
else:
    print("Leakage rate in test (unique-string basis): n/a (empty test split)")

Shared train/test strings: 22,550
Leakage rate in test (unique-string basis): 65.87%
